# DATAQUEST 5.0 — final raw-data experiment

This notebook trains a raw-state CatBoost anchor and one full-target TabM candidate from scratch. Five outer folds remain untouched by preprocessing, early stopping, blend selection, and threshold selection. If the candidate does not clear the frozen stability gates, `submission.csv` uses the fresh anchor.

Runtime target: Kaggle T4×2. On two GPUs, the independent models for each fold run concurrently; on one GPU they run sequentially. No pretrained weights are used.


In [ ]:
import importlib.metadata as metadata
import importlib.util
import subprocess
import sys

PINNED = {
    "tabm": ("tabm", "0.0.3"),
    "rtdl_num_embeddings": ("rtdl_num_embeddings", "0.0.12"),
}

def ensure_exact(import_name, dist_name, version):
    installed = None
    try:
        installed = metadata.version(dist_name)
    except metadata.PackageNotFoundError:
        pass
    if importlib.util.find_spec(import_name) is None or installed != version:
        print(f"Installing {dist_name}=={version} (found {installed})")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
            f"{dist_name}=={version}",
        ])

for module_name, (dist_name, version) in PINNED.items():
    ensure_exact(module_name, dist_name, version)

if importlib.util.find_spec("catboost") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
        "catboost==1.2.8",
    ])
print("Dependencies ready.")


In [ ]:
import gc
import hashlib
import json
import os
import platform
import random
import time
import warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy.optimize import minimize
from sklearn.metrics import cohen_kappa_score, log_loss, mean_squared_error, roc_auc_score
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit

import catboost
from catboost import CatBoostRegressor
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tabm import TabM
from rtdl_num_embeddings import PiecewiseLinearEmbeddings, compute_bins

warnings.filterwarnings("ignore")

CFG = {
    "seed": 20260910,
    "folds": 5,
    "calibration_fraction": 0.10,
    "cat_iterations": 3500,
    "cat_learning_rate": 0.035,
    "cat_depth": 8,
    "cat_l2": 5.0,
    "cat_patience": 180,
    "tabm_epochs": 70,
    "tabm_patience": 8,
    "tabm_batch": 1024,
    "tabm_predict_batch": 4096,
    "tabm_k": 16,
    "tabm_blocks": 2,
    "tabm_width": 256,
    "tabm_bins": 32,
    "tabm_num_embedding": 12,
    "tabm_lr": 0.002,
    "tabm_weight_decay": 3e-4,
    "cdf_weight": 0.10,
    "bootstrap_reps": 2000,
    "minimum_gain": 0.0015,
    "minimum_positive_folds": 4,
    "worst_fold_floor": -0.0015,
    "minimum_bootstrap_probability": 0.90,
    "minimum_auc12_delta": -0.0010,
    "maximum_false_tail_ratio": 1.03,
}
ALPHA_GRID = np.array([0.0, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.75, 1.0])
TARGET, ID_COL, HOUSEHOLD_COL = "life_satisfaction", "id", "household_id"
OUT = Path("/kaggle/working/oqdq5_final") if Path("/kaggle/working").exists() else Path("oqdq5_final")
OUT.mkdir(parents=True, exist_ok=True)

def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CFG["seed"])
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

GPU_COUNT = torch.cuda.device_count()
print("Python", platform.python_version())
print("torch", torch.__version__, "CUDA", torch.version.cuda, "GPUs", GPU_COUNT)
for device_index in range(GPU_COUNT):
    print(device_index, torch.cuda.get_device_name(device_index))
print(CFG)


## Raw inputs and respondent integrity

The file hashes identify the supplied competition release. Duplicate respondent rows must be identical before they are collapsed. The submission unit is the first-occurrence sequence of unique test respondent IDs; the provided template is also checked because its row IDs are a separate placeholder sequence.


In [ ]:
FILES = ["train_final.csv", "test_final.csv", "data_dictionary.csv", "sample_submission.csv"]
EXPECTED_HASHES = {
    "train_final.csv": "0ccc7f7aa3d9e7520d1edd0cae30c1ac119fcb4c508fdd7c9ebc871ceaee2056",
    "test_final.csv": "c7027da38aba915e6a9952d21c6cd960b22c172e0ccf3f9f570d63d07e008740",
    "data_dictionary.csv": "4ee1269e4e10fd62225916bb8eff1cd51930a211cb8488d6c97a5b4817e4f631",
    "sample_submission.csv": "3b27035c9910f24c7a0cf29f85c1382be423f58cda0524bbd7b0a24ab9e65abf",
}

def file_hash(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def discover_data_dir():
    candidates = []
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        candidates.extend(path.parent for path in kaggle_root.rglob("train_final.csv"))
    for local_root in (Path("data"), Path(".")):
        if (local_root / "train_final.csv").exists():
            candidates.append(local_root)
    valid = []
    for folder in sorted(set(candidates), key=lambda x: (len(str(x)), str(x))):
        if not all((folder / name).is_file() for name in FILES):
            continue
        hashes = {name: file_hash(folder / name) for name in FILES}
        if hashes == EXPECTED_HASHES:
            valid.append(folder)
    if not valid:
        raise FileNotFoundError(
            "The four competition CSV files were not found with the expected release hashes. "
            "Attach the competition dataset without re-exporting the CSV files."
        )
    return valid[0]

DATA_DIR = discover_data_dir()
train_raw = pd.read_csv(DATA_DIR / "train_final.csv", low_memory=False)
test_raw = pd.read_csv(DATA_DIR / "test_final.csv", low_memory=False)
data_dictionary = pd.read_csv(DATA_DIR / "data_dictionary.csv", low_memory=False)
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", low_memory=False)

assert train_raw.shape == (95609, 179), train_raw.shape
assert test_raw.shape == (23902, 178), test_raw.shape
assert list(sample_submission.columns) == [ID_COL, TARGET]
assert TARGET in train_raw and TARGET not in test_raw
assert set(train_raw[TARGET].dropna().astype(int).unique()) == {1, 2, 3, 4}
assert train_raw[ID_COL].notna().all() and test_raw[ID_COL].notna().all()

def assert_duplicate_identity(frame):
    value_columns = [column for column in frame.columns if column != ID_COL]
    row_hash = pd.util.hash_pandas_object(frame[value_columns], index=False)
    distinct = pd.DataFrame({ID_COL: frame[ID_COL].to_numpy(), "row_hash": row_hash.to_numpy()})
    maximum_versions = int(distinct.groupby(ID_COL, sort=False)["row_hash"].nunique().max())
    if maximum_versions != 1:
        raise ValueError("At least one repeated respondent ID has conflicting row content.")

assert_duplicate_identity(train_raw)
assert_duplicate_identity(test_raw)
assert int(train_raw.groupby(ID_COL)[TARGET].nunique().max()) == 1

train = train_raw.drop_duplicates(ID_COL, keep="first").reset_index(drop=True)
test = test_raw.drop_duplicates(ID_COL, keep="first").reset_index(drop=True)
assert train.shape[0] == 94663 and test.shape[0] == 23666
assert train[ID_COL].is_unique and test[ID_COL].is_unique
assert set(train[ID_COL]).isdisjoint(set(test[ID_COL]))

CONTRACT = {
    "raw_train_rows": len(train_raw),
    "unique_train_ids": len(train),
    "raw_test_rows": len(test_raw),
    "unique_test_ids": len(test),
    "sample_rows": len(sample_submission),
    "sample_ids_equal_raw_test_ids": bool(np.array_equal(sample_submission[ID_COL], test_raw[ID_COL])),
    "submission_unit": "unique_test_id_first_occurrence",
}
print("DATA_DIR", DATA_DIR)
print(CONTRACT)
print(train[TARGET].value_counts().sort_index())


## Raw-state representation

Low-cardinality response values remain feature-specific categorical states. Numeric fields keep a robust numeric copy; actual missing values and documented special values also receive explicit state tokens. Every learned vocabulary, scale, and numeric bin is fit inside a fold.


In [ ]:
RAW_FEATURES = [column for column in test.columns if column not in {ID_COL, HOUSEHOLD_COL}]
NUMERIC_REQUESTED = [
    "survey_weight", "income_to_poverty_ratio", "age", "height_inches", "weight_pounds",
    "hours_worked_per_week", "days_missed_work", "emergency_room_visits",
    "years_in_current_home", "children_in_family", "children_in_household",
    "adults_in_family", "adults_in_household", "cancer_types_count",
]
NUMERIC_RAW = [column for column in NUMERIC_REQUESTED if column in RAW_FEATURES]
SPECIAL_NUMERIC_CODES = {
    "weight_pounds": {996, 997, 999},
    "height_inches": {96, 97, 98, 99},
    "age": {97, 99},
    "hours_worked_per_week": {990},
    "days_missed_work": {997, 998, 999},
}
CAT_RAW = [column for column in RAW_FEATURES if column not in NUMERIC_RAW]

def make_state_frame(frame):
    columns = {}
    for column in CAT_RAW:
        columns[column] = frame[column].astype("string").fillna("__NA__").astype(str)
    for column in NUMERIC_RAW:
        raw = pd.to_numeric(frame[column], errors="coerce")
        state = pd.Series("__OBS__", index=frame.index, dtype="string")
        state.loc[raw.isna()] = "__NA__"
        special = SPECIAL_NUMERIC_CODES.get(column, set())
        for value in sorted(special):
            state.loc[raw.eq(value)] = f"__CODE_{value}__"
        clean = raw.mask(raw.isin(special)).astype(np.float32)
        columns[column] = clean
        columns[f"{column}__state"] = state.astype(str)
    columns["_missing_count"] = frame[RAW_FEATURES].isna().sum(axis=1).astype(np.float32)
    return pd.DataFrame(columns, index=frame.index)

train_x = make_state_frame(train)
test_x = make_state_frame(test)
NUM_FEATURES = NUMERIC_RAW + ["_missing_count"]
CAT_FEATURES = CAT_RAW + [f"{column}__state" for column in NUMERIC_RAW]
MODEL_FEATURES = CAT_FEATURES + NUM_FEATURES
train_x = train_x[MODEL_FEATURES]
test_x = test_x[MODEL_FEATURES]
CB_CAT_INDICES = [train_x.columns.get_loc(column) for column in CAT_FEATURES]

assert train_x.shape[1] == test_x.shape[1]
assert train_x.columns.tolist() == test_x.columns.tolist()
assert not ({ID_COL, HOUSEHOLD_COL, TARGET} & set(train_x.columns))
print("features", len(MODEL_FEATURES), "categorical", len(CAT_FEATURES), "numeric", len(NUM_FEATURES))

class FoldPreprocessor:
    def fit(self, frame):
        self.category_maps = {}
        self.cardinalities = []
        for column in CAT_FEATURES:
            values = pd.Index(frame[column].astype(str).unique())
            mapping = {value: index + 1 for index, value in enumerate(values)}
            self.category_maps[column] = mapping
            self.cardinalities.append(len(mapping) + 1)
        self.numeric_stats = {}
        self.numeric_keep = []
        for column in NUM_FEATURES:
            values = pd.to_numeric(frame[column], errors="coerce").astype(float)
            median = float(values.median()) if values.notna().any() else 0.0
            q1 = float(values.quantile(0.25)) if values.notna().any() else median
            q3 = float(values.quantile(0.75)) if values.notna().any() else median + 1.0
            scale = max(q3 - q1, 1e-3)
            transformed = ((values.fillna(median) - median) / scale).clip(-12, 12)
            if transformed.nunique(dropna=False) > 1:
                self.numeric_keep.append(column)
                self.numeric_stats[column] = (median, scale)
        return self

    def transform(self, frame):
        cat_arrays = []
        for column in CAT_FEATURES:
            encoded = frame[column].astype(str).map(self.category_maps[column]).fillna(0)
            cat_arrays.append(encoded.astype(np.int64).to_numpy())
        x_cat = np.column_stack(cat_arrays).astype(np.int64)
        num_arrays = []
        for column in self.numeric_keep:
            median, scale = self.numeric_stats[column]
            values = pd.to_numeric(frame[column], errors="coerce").astype(float)
            encoded = ((values.fillna(median) - median) / scale).clip(-12, 12)
            num_arrays.append(encoded.astype(np.float32).to_numpy())
        x_num = np.column_stack(num_arrays).astype(np.float32)
        return x_num, x_cat


## Metrics, fold-pure calibration, and fixed gates

Each outer fold gets a calibration subset from its training partition. Cut points and the small blend grid use only that subset. The scored fold is never used to choose them.


In [ ]:
WEIGHT_MATRIX = ((np.arange(4)[:, None] - np.arange(4)[None, :]) ** 2) / 9.0

def qwk_fast(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.int64) - 1
    y_pred = np.asarray(y_pred, dtype=np.int64) - 1
    confusion = np.bincount(y_true * 4 + y_pred, minlength=16).reshape(4, 4).astype(float)
    n = confusion.sum()
    observed = float((WEIGHT_MATRIX * confusion).sum() / n)
    expected_matrix = np.outer(confusion.sum(axis=1), confusion.sum(axis=0)) / n
    expected = float((WEIGHT_MATRIX * expected_matrix).sum() / n)
    if expected <= 0:
        return 1.0 if observed <= 0 else 0.0
    return 1.0 - observed / expected

def apply_cuts(score, cuts):
    return np.digitize(np.asarray(score, dtype=float), np.sort(np.asarray(cuts, dtype=float))) + 1

def fit_cuts(y_true, score):
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    starts = [(1.50, 2.00, 2.50), (1.40, 1.90, 2.40), (1.60, 2.10, 2.65)]

    def objective(raw_cuts):
        cuts = np.sort(raw_cuts)
        if np.min(np.diff(cuts)) < 0.025:
            return 2.0
        return -qwk_fast(y_true, apply_cuts(score, cuts))

    best = None
    for start in starts:
        fitted = minimize(
            objective, np.asarray(start, dtype=float), method="Nelder-Mead",
            options={"maxiter": 500, "xatol": 1e-5, "fatol": 1e-6},
        )
        cuts = np.sort(fitted.x)
        value = -objective(cuts)
        if best is None or value > best[0]:
            best = (value, cuts)
    return best[1]

def boundary_auc(y_true, score, boundary):
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    if boundary == "1v2":
        mask = y_true <= 2
        target = (y_true[mask] == 2).astype(int)
    elif boundary == "tail":
        mask = np.ones(len(y_true), dtype=bool)
        target = (y_true >= 3).astype(int)
    elif boundary == "3v4":
        mask = y_true >= 3
        target = (y_true[mask] == 4).astype(int)
    else:
        raise ValueError(boundary)
    return float(roc_auc_score(target, score[mask]))

def transition_counts(y_true, prediction):
    y_true = np.asarray(y_true, dtype=int)
    prediction = np.asarray(prediction, dtype=int)
    squared = (y_true - prediction) ** 2
    masks = {
        "1_to_2": (y_true == 1) & (prediction == 2),
        "2_to_1": (y_true == 2) & (prediction == 1),
        "false_tail": (y_true <= 2) & (prediction >= 3),
        "missed_tail": (y_true >= 3) & (prediction <= 2),
        "3_4": ((y_true == 3) & (prediction == 4)) | ((y_true == 4) & (prediction == 3)),
    }
    return {
        name: {"rows": int(mask.sum()), "weighted_loss": int(squared[mask].sum())}
        for name, mask in masks.items()
    }

def score_metrics(y_true, score, prediction):
    return {
        "qwk": qwk_fast(y_true, prediction),
        "accuracy": float(np.mean(np.asarray(y_true) == np.asarray(prediction))),
        "rmse": float(mean_squared_error(y_true, score) ** 0.5),
        "auc_1v2": boundary_auc(y_true, score, "1v2"),
        "auc_tail": boundary_auc(y_true, score, "tail"),
        "auc_3v4": boundary_auc(y_true, score, "3v4"),
    }

def tune_blend(y_cal, anchor_score, tabm_score):
    best = None
    for alpha in ALPHA_GRID:
        score = (1.0 - alpha) * anchor_score + alpha * tabm_score
        cuts = fit_cuts(y_cal, score)
        value = qwk_fast(y_cal, apply_cuts(score, cuts))
        record = (value, -float(alpha), float(alpha), cuts)
        if best is None or record[:2] > best[:2]:
            best = record
    return {"calibration_qwk": best[0], "alpha": best[2], "cuts": best[3]}

def make_strata(frame):
    year = frame.get("survey_year", pd.Series("NA", index=frame.index)).astype("string").fillna("NA")
    combined = frame[TARGET].astype(str) + "_" + year.astype(str)
    return combined if combined.value_counts().min() >= CFG["folds"] else frame[TARGET].astype(str)

def high_burden_mask(frame):
    def numeric(column):
        return pd.to_numeric(frame[column], errors="coerce") if column in frame else pd.Series(np.nan, index=frame.index)
    severe = pd.DataFrame({
        "health": numeric("general_health_status").between(4, 5),
        "depression": numeric("depression_frequency").between(1, 2),
        "anxiety": numeric("anxiety_frequency").between(1, 2),
        "food": numeric("food_security_4cat").between(3, 4),
        "cost": numeric("worry_about_medical_costs").eq(1),
        "work": numeric("work_limited_by_health").eq(1),
        "social": numeric("difficulty_social_activities").eq(1),
    })
    return severe.sum(axis=1).ge(3).to_numpy()

_check_y = np.array([1, 1, 2, 2, 3, 4, 4])
_check_p = np.array([1, 2, 2, 3, 3, 3, 4])
assert abs(qwk_fast(_check_y, _check_p) - cohen_kappa_score(_check_y, _check_p, weights="quadratic")) < 1e-12
assert np.array_equal(apply_cuts([1.1, 1.6, 2.1, 3.0], [1.5, 2.0, 2.5]), [1, 2, 3, 4])
print("Metric checks passed.")


## Models

CatBoost minimizes RMSE on the ordinal label. TabM minimizes four-class cross-entropy plus a fixed small cumulative-distribution penalty; early stopping uses that same proper held-out loss rather than repeatedly optimizing QWK.


In [ ]:
def numpy_tabm_loss(y_zero, probabilities):
    labels = np.asarray(y_zero, dtype=int)
    probabilities = np.clip(np.asarray(probabilities, dtype=float), 1e-7, 1.0)
    ce = log_loss(labels, probabilities, labels=[0, 1, 2, 3])
    one_hot = np.eye(4, dtype=float)[labels]
    cdf = np.mean((np.cumsum(probabilities, axis=1)[:, :-1] - np.cumsum(one_hot, axis=1)[:, :-1]) ** 2)
    return float(ce + CFG["cdf_weight"] * cdf)

def tabm_loss(logits, labels):
    batch, members, classes = logits.shape
    expanded = labels[:, None].expand(batch, members).reshape(-1)
    flat_logits = logits.reshape(-1, classes)
    ce = F.cross_entropy(flat_logits, expanded)
    probabilities = torch.softmax(logits, dim=-1)
    one_hot = F.one_hot(labels, num_classes=4).to(probabilities.dtype)[:, None, :].expand_as(probabilities)
    cdf = F.mse_loss(probabilities.cumsum(dim=-1)[..., :-1], one_hot.cumsum(dim=-1)[..., :-1])
    return ce + CFG["cdf_weight"] * cdf

def tabm_predict(model, x_num, x_cat, device):
    dataset = TensorDataset(torch.from_numpy(x_num), torch.from_numpy(x_cat))
    loader = DataLoader(
        dataset, batch_size=CFG["tabm_predict_batch"], shuffle=False, num_workers=0,
        pin_memory=True,
    )
    predictions = []
    model.eval()
    with torch.no_grad():
        for numeric, categorical in loader:
            numeric = numeric.to(device, non_blocking=True)
            categorical = categorical.to(device, non_blocking=True)
            logits = model(numeric, categorical)
            probabilities = torch.softmax(logits, dim=-1).mean(dim=1)
            predictions.append(probabilities.float().cpu().numpy())
    return np.concatenate(predictions)

def train_tabm_fold(fit_index, cal_index, val_index, fold, device_index):
    started = time.time()
    seed = 20263000 + fold
    seed_everything(seed)
    torch.cuda.set_device(device_index)
    device = torch.device(f"cuda:{device_index}")
    preprocessor = FoldPreprocessor().fit(train_x.iloc[fit_index])
    x_fit_num, x_fit_cat = preprocessor.transform(train_x.iloc[fit_index])
    x_cal_num, x_cal_cat = preprocessor.transform(train_x.iloc[cal_index])
    x_val_num, x_val_cat = preprocessor.transform(train_x.iloc[val_index])
    x_test_num, x_test_cat = preprocessor.transform(test_x)

    bins = compute_bins(torch.from_numpy(x_fit_num), n_bins=CFG["tabm_bins"])
    embeddings = PiecewiseLinearEmbeddings(
        bins, CFG["tabm_num_embedding"], activation=False, version="B"
    )
    model = TabM.make(
        n_num_features=x_fit_num.shape[1],
        cat_cardinalities=preprocessor.cardinalities,
        num_embeddings=embeddings,
        d_out=4,
        k=CFG["tabm_k"],
        n_blocks=CFG["tabm_blocks"],
        d_block=CFG["tabm_width"],
    ).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CFG["tabm_lr"], weight_decay=CFG["tabm_weight_decay"]
    )
    scaler = torch.cuda.amp.GradScaler(enabled=True)
    y_fit_zero = train.iloc[fit_index][TARGET].to_numpy(dtype=np.int64) - 1
    y_cal_zero = train.iloc[cal_index][TARGET].to_numpy(dtype=np.int64) - 1
    generator = torch.Generator().manual_seed(seed)
    dataset = TensorDataset(
        torch.from_numpy(x_fit_num), torch.from_numpy(x_fit_cat), torch.from_numpy(y_fit_zero)
    )
    loader = DataLoader(
        dataset, batch_size=CFG["tabm_batch"], shuffle=True, generator=generator,
        num_workers=0, pin_memory=True,
    )

    best_loss, best_state, best_epoch, stale = float("inf"), None, 0, 0
    for epoch in range(1, CFG["tabm_epochs"] + 1):
        model.train()
        epoch_losses = []
        for numeric, categorical, labels in loader:
            numeric = numeric.to(device, non_blocking=True)
            categorical = categorical.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=True):
                loss = tabm_loss(model(numeric, categorical), labels)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"TabM fold {fold} produced non-finite loss at epoch {epoch}.")
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            epoch_losses.append(float(loss.detach().cpu()))

        cal_probabilities = tabm_predict(model, x_cal_num, x_cal_cat, device)
        cal_loss = numpy_tabm_loss(y_cal_zero, cal_probabilities)
        if cal_loss < best_loss - 1e-5:
            best_loss = cal_loss
            best_epoch = epoch
            best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
        if epoch == 1 or epoch % 10 == 0:
            print(f"fold {fold} TabM epoch {epoch}: train={np.mean(epoch_losses):.5f} cal={cal_loss:.5f}")
        if stale >= CFG["tabm_patience"]:
            break

    if best_state is None:
        raise RuntimeError(f"TabM fold {fold} did not produce a finite checkpoint.")
    model.load_state_dict(best_state)
    model.to(device)
    outputs = {
        "cal_probabilities": tabm_predict(model, x_cal_num, x_cal_cat, device),
        "val_probabilities": tabm_predict(model, x_val_num, x_val_cat, device),
        "test_probabilities": tabm_predict(model, x_test_num, x_test_cat, device),
        "best_epoch": best_epoch,
        "best_calibration_loss": best_loss,
        "seconds": time.time() - started,
        "numeric_features_kept": list(preprocessor.numeric_keep),
        "category_cardinalities": list(preprocessor.cardinalities),
    }
    del model, best_state, loader, dataset
    torch.cuda.empty_cache()
    gc.collect()
    return outputs

def train_catboost_fold(fit_index, cal_index, val_index, fold, device_index):
    started = time.time()
    model = CatBoostRegressor(
        loss_function="RMSE", eval_metric="RMSE", iterations=CFG["cat_iterations"],
        learning_rate=CFG["cat_learning_rate"], depth=CFG["cat_depth"],
        l2_leaf_reg=CFG["cat_l2"], random_seed=20262000 + fold,
        task_type="GPU", devices=str(device_index), allow_writing_files=False,
        verbose=250,
    )
    model.fit(
        train_x.iloc[fit_index], train.iloc[fit_index][TARGET],
        cat_features=CB_CAT_INDICES,
        eval_set=(train_x.iloc[cal_index], train.iloc[cal_index][TARGET]),
        early_stopping_rounds=CFG["cat_patience"],
    )
    output = {
        "cal_score": model.predict(train_x.iloc[cal_index]).astype(float),
        "val_score": model.predict(train_x.iloc[val_index]).astype(float),
        "test_score": model.predict(test_x).astype(float),
        "best_iteration": int(model.get_best_iteration()),
        "seconds": time.time() - started,
    }
    del model
    gc.collect()
    return output

_synthetic_logits = torch.tensor([[[2.0, 1.0, 0.0, -1.0], [1.0, 2.0, 0.0, -1.0]]])
_synthetic_label = torch.tensor([0])
assert torch.isfinite(tabm_loss(_synthetic_logits, _synthetic_label))
print("Model-loss check passed.")


## Five-fold run

This is the only expensive cell. It saves fold-pure predictions for the anchor, candidate, and calibration-selected blend.


In [ ]:
if GPU_COUNT < 1:
    raise RuntimeError("Enable a Kaggle GPU accelerator. The full TabM experiment requires one CUDA device.")

y = train[TARGET].to_numpy(dtype=int)
outer = StratifiedKFold(n_splits=CFG["folds"], shuffle=True, random_state=CFG["seed"])
outer_folds = list(outer.split(train, make_strata(train)))

oof_anchor_score = np.full(len(train), np.nan)
oof_tabm_score = np.full(len(train), np.nan)
oof_blend_score = np.full(len(train), np.nan)
oof_anchor_pred = np.zeros(len(train), dtype=np.int8)
oof_tabm_pred = np.zeros(len(train), dtype=np.int8)
oof_blend_pred = np.zeros(len(train), dtype=np.int8)
oof_tabm_probability = np.full((len(train), 4), np.nan)
fold_assignment = np.full(len(train), -1, dtype=np.int8)

fold_metric_rows = []
calibration_rows = []
fold_manifest = []
test_anchor_folds = []
test_tabm_probability_folds = []
selected_alphas = []
anchor_cut_folds = []
tabm_cut_folds = []
blend_cut_folds = []

for fold, (outer_train, outer_val) in enumerate(outer_folds):
    print(f"\n{'=' * 90}\nOUTER FOLD {fold}\n{'=' * 90}")
    inner_frame = train.iloc[outer_train]
    splitter = StratifiedShuffleSplit(
        n_splits=1, test_size=CFG["calibration_fraction"], random_state=20261000 + fold
    )
    fit_relative, cal_relative = next(splitter.split(inner_frame, make_strata(inner_frame)))
    fit_index = np.asarray(outer_train)[fit_relative]
    cal_index = np.asarray(outer_train)[cal_relative]

    if GPU_COUNT >= 2:
        with ThreadPoolExecutor(max_workers=2) as executor:
            tabm_future = executor.submit(train_tabm_fold, fit_index, cal_index, outer_val, fold, 0)
            cat_future = executor.submit(train_catboost_fold, fit_index, cal_index, outer_val, fold, 1)
            try:
                tabm_output = tabm_future.result()
            except Exception as error:
                raise RuntimeError(f"TabM failed in outer fold {fold} on CUDA device 0: {error}") from error
            try:
                cat_output = cat_future.result()
            except Exception as first_error:
                print(f"CatBoost device 1 failed in fold {fold}; retrying sequentially on device 0: {first_error!r}")
                try:
                    cat_output = train_catboost_fold(fit_index, cal_index, outer_val, fold, 0)
                except Exception as second_error:
                    raise RuntimeError(f"CatBoost failed in outer fold {fold} on both device paths: {second_error}") from second_error
    else:
        try:
            cat_output = train_catboost_fold(fit_index, cal_index, outer_val, fold, 0)
            tabm_output = train_tabm_fold(fit_index, cal_index, outer_val, fold, 0)
        except Exception as error:
            raise RuntimeError(f"Sequential single-GPU training failed in outer fold {fold}: {error}") from error

    class_values = np.arange(1, 5, dtype=float)
    tabm_cal_score = tabm_output["cal_probabilities"] @ class_values
    tabm_val_score = tabm_output["val_probabilities"] @ class_values
    tabm_test_score = tabm_output["test_probabilities"] @ class_values
    anchor_cal_score = cat_output["cal_score"]
    anchor_val_score = cat_output["val_score"]
    y_cal = y[cal_index]
    y_val = y[outer_val]

    anchor_cuts = fit_cuts(y_cal, anchor_cal_score)
    tabm_cuts = fit_cuts(y_cal, tabm_cal_score)
    blend_tuning = tune_blend(y_cal, anchor_cal_score, tabm_cal_score)
    alpha = blend_tuning["alpha"]
    blend_cuts = blend_tuning["cuts"]
    blend_val_score = (1.0 - alpha) * anchor_val_score + alpha * tabm_val_score

    anchor_val_pred = apply_cuts(anchor_val_score, anchor_cuts)
    tabm_val_pred = apply_cuts(tabm_val_score, tabm_cuts)
    blend_val_pred = apply_cuts(blend_val_score, blend_cuts)

    oof_anchor_score[outer_val] = anchor_val_score
    oof_tabm_score[outer_val] = tabm_val_score
    oof_blend_score[outer_val] = blend_val_score
    oof_anchor_pred[outer_val] = anchor_val_pred
    oof_tabm_pred[outer_val] = tabm_val_pred
    oof_blend_pred[outer_val] = blend_val_pred
    oof_tabm_probability[outer_val] = tabm_output["val_probabilities"]
    fold_assignment[outer_val] = fold

    for branch, score, prediction in [
        ("anchor", anchor_val_score, anchor_val_pred),
        ("tabm", tabm_val_score, tabm_val_pred),
        ("blend", blend_val_score, blend_val_pred),
    ]:
        metrics = score_metrics(y_val, score, prediction)
        metrics.update({"fold": fold, "branch": branch, "alpha": alpha if branch == "blend" else np.nan})
        fold_metric_rows.append(metrics)

    anchor_qwk = fold_metric_rows[-3]["qwk"]
    for row in fold_metric_rows[-3:]:
        row["delta_qwk_vs_anchor"] = row["qwk"] - anchor_qwk
    fold_metric_rows[-1]["score_correlation_vs_anchor"] = float(np.corrcoef(anchor_val_score, blend_val_score)[0, 1])
    fold_metric_rows[-1]["residual_correlation_vs_anchor"] = float(
        np.corrcoef(y_val - anchor_val_score, y_val - blend_val_score)[0, 1]
    )

    for branch, cuts, cal_score in [
        ("anchor", anchor_cuts, anchor_cal_score),
        ("tabm", tabm_cuts, tabm_cal_score),
        ("blend", blend_cuts, (1.0 - alpha) * anchor_cal_score + alpha * tabm_cal_score),
    ]:
        calibration_rows.append({
            "fold": fold, "branch": branch, "alpha": alpha if branch == "blend" else np.nan,
            "cut_1": float(cuts[0]), "cut_2": float(cuts[1]), "cut_3": float(cuts[2]),
            "calibration_qwk": qwk_fast(y_cal, apply_cuts(cal_score, cuts)),
            "fit_rows": len(fit_index), "calibration_rows": len(cal_index), "validation_rows": len(outer_val),
        })

    selected_alphas.append(alpha)
    anchor_cut_folds.append(anchor_cuts)
    tabm_cut_folds.append(tabm_cuts)
    blend_cut_folds.append(blend_cuts)
    test_anchor_folds.append(cat_output["test_score"])
    test_tabm_probability_folds.append(tabm_output["test_probabilities"])
    fold_manifest.append({
        "fold": fold, "fit_rows": len(fit_index), "calibration_rows": len(cal_index),
        "validation_rows": len(outer_val), "cat_best_iteration": cat_output["best_iteration"],
        "cat_seconds": cat_output["seconds"], "tabm_best_epoch": tabm_output["best_epoch"],
        "tabm_best_calibration_loss": tabm_output["best_calibration_loss"],
        "tabm_seconds": tabm_output["seconds"], "alpha": alpha,
        "anchor_cuts": anchor_cuts.tolist(), "tabm_cuts": tabm_cuts.tolist(),
        "blend_cuts": blend_cuts.tolist(),
    })
    print(pd.DataFrame(fold_metric_rows[-3:]).to_string(index=False))
    del cat_output, tabm_output
    torch.cuda.empty_cache()
    gc.collect()

assert np.isfinite(oof_anchor_score).all()
assert np.isfinite(oof_tabm_score).all()
assert np.isfinite(oof_blend_score).all()
assert np.isfinite(oof_tabm_probability).all()
assert (fold_assignment >= 0).all()


## Paired decision, error audit, and deployment aggregation

Outer-fold predictions decide whether the blend is admitted. Test aggregation uses only calibration choices made within outer-training partitions.


In [ ]:
fold_metrics = pd.DataFrame(fold_metric_rows)
calibration_audit = pd.DataFrame(calibration_rows)

pooled = {}
for branch, score, prediction in [
    ("anchor", oof_anchor_score, oof_anchor_pred),
    ("tabm", oof_tabm_score, oof_tabm_pred),
    ("blend", oof_blend_score, oof_blend_pred),
]:
    pooled[branch] = score_metrics(y, score, prediction)

anchor_by_fold = fold_metrics.query("branch == 'anchor'").sort_values("fold")["qwk"].to_numpy()
blend_by_fold = fold_metrics.query("branch == 'blend'").sort_values("fold")["qwk"].to_numpy()
fold_deltas = blend_by_fold - anchor_by_fold

rng = np.random.default_rng(20264000)
bootstrap_delta = np.empty(CFG["bootstrap_reps"], dtype=float)
for iteration in range(CFG["bootstrap_reps"]):
    sampled = rng.integers(0, len(y), len(y))
    bootstrap_delta[iteration] = (
        qwk_fast(y[sampled], oof_blend_pred[sampled])
        - qwk_fast(y[sampled], oof_anchor_pred[sampled])
    )
bootstrap = {
    "repetitions": CFG["bootstrap_reps"],
    "probability_gain_positive": float(np.mean(bootstrap_delta > 0)),
    "mean_delta": float(np.mean(bootstrap_delta)),
    "ci_025": float(np.quantile(bootstrap_delta, 0.025)),
    "ci_975": float(np.quantile(bootstrap_delta, 0.975)),
}

anchor_transitions = transition_counts(y, oof_anchor_pred)
blend_transitions = transition_counts(y, oof_blend_pred)
false_tail_ratio = (
    blend_transitions["false_tail"]["weighted_loss"]
    / max(anchor_transitions["false_tail"]["weighted_loss"], 1)
)
pooled_gain = pooled["blend"]["qwk"] - pooled["anchor"]["qwk"]
auc12_delta = pooled["blend"]["auc_1v2"] - pooled["anchor"]["auc_1v2"]

promotion_checks = {
    "pooled_gain": bool(pooled_gain >= CFG["minimum_gain"]),
    "positive_folds": bool(int((fold_deltas > 0).sum()) >= CFG["minimum_positive_folds"]),
    "worst_fold": bool(float(fold_deltas.min()) >= CFG["worst_fold_floor"]),
    "bootstrap_probability": bool(bootstrap["probability_gain_positive"] >= CFG["minimum_bootstrap_probability"]),
    "auc_1v2": bool(auc12_delta >= CFG["minimum_auc12_delta"]),
    "false_tail": bool(false_tail_ratio <= CFG["maximum_false_tail_ratio"]),
}
PROMOTED = all(promotion_checks.values())
FINAL_BRANCH = "blend" if PROMOTED else "anchor"

mean_anchor_test = np.mean(np.stack(test_anchor_folds), axis=0)
mean_tabm_probability = np.mean(np.stack(test_tabm_probability_folds), axis=0)
mean_tabm_test = mean_tabm_probability @ np.arange(1, 5, dtype=float)
if PROMOTED:
    deployment_alpha = float(np.median(selected_alphas))
    deployment_cuts = np.median(np.stack(blend_cut_folds), axis=0)
    final_test_score = (1.0 - deployment_alpha) * mean_anchor_test + deployment_alpha * mean_tabm_test
    final_oof_score, final_oof_pred = oof_blend_score, oof_blend_pred
else:
    deployment_alpha = 0.0
    deployment_cuts = np.median(np.stack(anchor_cut_folds), axis=0)
    final_test_score = mean_anchor_test
    final_oof_score, final_oof_pred = oof_anchor_score, oof_anchor_pred
final_test_prediction = apply_cuts(final_test_score, deployment_cuts).astype(np.int8)

confusion_transition_rows = []
for branch, prediction in [("anchor", oof_anchor_pred), ("tabm", oof_tabm_pred), ("blend", oof_blend_pred)]:
    confusion = np.bincount((y - 1) * 4 + (prediction - 1), minlength=16).reshape(4, 4)
    for true_label in range(1, 5):
        for predicted_label in range(1, 5):
            confusion_transition_rows.append({
                "branch": branch, "record_type": "confusion", "name": f"{true_label}_to_{predicted_label}",
                "rows": int(confusion[true_label - 1, predicted_label - 1]), "weighted_loss": int(
                    confusion[true_label - 1, predicted_label - 1] * (true_label - predicted_label) ** 2
                ),
            })
    for name, values in transition_counts(y, prediction).items():
        confusion_transition_rows.append({"branch": branch, "record_type": "transition", "name": name, **values})
confusion_transition_audit = pd.DataFrame(confusion_transition_rows)

burden = high_burden_mask(train)
subgroup_rows = []
for branch, score, prediction in [
    ("anchor", oof_anchor_score, oof_anchor_pred),
    ("tabm", oof_tabm_score, oof_tabm_pred),
    ("blend", oof_blend_score, oof_blend_pred),
]:
    for group_name, mask in [("all", np.ones(len(y), dtype=bool)), ("high_burden", burden), ("other", ~burden)]:
        row = score_metrics(y[mask], score[mask], prediction[mask])
        row.update({"branch": branch, "group": group_name, "rows": int(mask.sum())})
        subgroup_rows.append(row)
subgroup_metrics = pd.DataFrame(subgroup_rows)

print("pooled", json.dumps(pooled, indent=2))
print("fold deltas", fold_deltas)
print("bootstrap", bootstrap)
print("promotion checks", promotion_checks)
print("FINAL_BRANCH", FINAL_BRANCH, "alpha", deployment_alpha, "cuts", deployment_cuts)


## Save and verify artifacts

All output files are produced by this run. `submission.csv` uses the unique respondent contract and preserves first appearance order from the test table.


In [ ]:
oof_predictions = pd.DataFrame({
    ID_COL: train[ID_COL].to_numpy(),
    TARGET: y,
    "fold": fold_assignment,
    "high_burden": burden.astype(np.int8),
    "anchor_score": oof_anchor_score,
    "anchor_prediction": oof_anchor_pred,
    "tabm_score": oof_tabm_score,
    "tabm_prediction": oof_tabm_pred,
    "blend_score": oof_blend_score,
    "blend_prediction": oof_blend_pred,
    "final_score": final_oof_score,
    "final_prediction": final_oof_pred,
})
for class_index in range(4):
    oof_predictions[f"tabm_probability_{class_index + 1}"] = oof_tabm_probability[:, class_index]

test_scores = pd.DataFrame({
    ID_COL: test[ID_COL].to_numpy(),
    "anchor_score": mean_anchor_test,
    "tabm_score": mean_tabm_test,
    "final_score": final_test_score,
})
for class_index in range(4):
    test_scores[f"tabm_probability_{class_index + 1}"] = mean_tabm_probability[:, class_index]

submission = pd.DataFrame({ID_COL: test[ID_COL].to_numpy(), TARGET: final_test_prediction.astype(int)})
assert list(submission.columns) == list(sample_submission.columns)
assert len(submission) == test[ID_COL].nunique() == 23666
assert submission[ID_COL].is_unique
assert submission[ID_COL].tolist() == test_raw[ID_COL].drop_duplicates(keep="first").tolist()
assert submission[TARGET].isin([1, 2, 3, 4]).all()
assert np.issubdtype(submission[TARGET].dtype, np.integer)
assert np.isfinite(test_scores.drop(columns=[ID_COL]).to_numpy()).all()

def package_version(distribution):
    try:
        return metadata.version(distribution)
    except metadata.PackageNotFoundError:
        return None

manifest = {
    "raw_file_hashes": EXPECTED_HASHES,
    "contract": CONTRACT,
    "config": CFG,
    "alpha_grid": ALPHA_GRID.tolist(),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "packages": {
        "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
        "scikit_learn": package_version("scikit-learn"), "catboost": catboost.__version__,
        "torch": torch.__version__, "tabm": package_version("tabm"),
        "rtdl_num_embeddings": package_version("rtdl_num_embeddings"),
    },
    "cuda": {
        "count": GPU_COUNT,
        "devices": [torch.cuda.get_device_name(index) for index in range(GPU_COUNT)],
    },
    "raw_features": RAW_FEATURES,
    "categorical_features": CAT_FEATURES,
    "numeric_features": NUM_FEATURES,
    "special_numeric_codes": {name: sorted(values) for name, values in SPECIAL_NUMERIC_CODES.items()},
    "folds": fold_manifest,
}
summary = {
    "final_branch": FINAL_BRANCH,
    "candidate_promoted": PROMOTED,
    "promotion_checks": promotion_checks,
    "pooled_metrics": pooled,
    "pooled_blend_gain": pooled_gain,
    "fold_blend_deltas": fold_deltas.tolist(),
    "auc_1v2_delta": auc12_delta,
    "false_tail_weighted_loss_ratio": false_tail_ratio,
    "bootstrap": bootstrap,
    "deployment_alpha": deployment_alpha,
    "deployment_cuts": deployment_cuts.tolist(),
    "submission_rows": len(submission),
    "submission_label_counts": submission[TARGET].value_counts().sort_index().to_dict(),
}

paths = {
    "oof": OUT / "oof_predictions.csv",
    "fold_metrics": OUT / "fold_metrics.csv",
    "calibration": OUT / "threshold_calibration_audit.csv",
    "confusion_transitions": OUT / "confusion_and_transition_audit.csv",
    "subgroups": OUT / "subgroup_metrics.csv",
    "test_scores": OUT / "test_scores.csv",
    "manifest": OUT / "feature_model_manifest.json",
    "submission": OUT / "submission.csv",
    "summary": OUT / "summary.json",
}
oof_predictions.to_csv(paths["oof"], index=False)
fold_metrics.to_csv(paths["fold_metrics"], index=False)
calibration_audit.to_csv(paths["calibration"], index=False)
confusion_transition_audit.to_csv(paths["confusion_transitions"], index=False)
subgroup_metrics.to_csv(paths["subgroups"], index=False)
test_scores.to_csv(paths["test_scores"], index=False)
submission.to_csv(paths["submission"], index=False)
with open(paths["manifest"], "w") as handle:
    json.dump(manifest, handle, indent=2)
with open(paths["summary"], "w") as handle:
    json.dump(summary, handle, indent=2)

reloaded = pd.read_csv(paths["submission"])
assert reloaded.equals(submission)
for path in paths.values():
    assert path.is_file() and path.stat().st_size > 0
print("Saved verified artifacts:")
for name, path in paths.items():
    print(name, path)
